# 05 — Add a new ingest market

Automate onboarding when adding a new country/route or marketplace source.

Uses `notification_rake.tools.market_setup` to:

1. Define a **MarketSpec** (route, country, sources)
2. Generate SQL migration + Python registry snippets
3. Register sources in `listings.source` (optional, requires DB)
4. Write review artifacts to `notebooks/out/market_<slug>/`
5. Audit what's still missing in the codebase

Run: `docker compose --profile dev-tools up -d jupyter` → open this notebook.

After running: copy snippets from the output folder into repo files, implement fetch logic, run `make test`.

In [ ]:
from notification_rake.tools.market_setup import (
    MarketSource,
    MarketSpec,
    audit_market,
    plan_market,
    print_plan_summary,
    register_sources,
    write_artifacts,
)
from notification_rake.config import settings
from notification_rake.models.ingest_routes import list_routes

print("Current ingest routes:")
for r in list_routes():
    print(f"  {r.slug:14} {r.label:20} {sorted(r.sources)}")

## 1. Define your market

Edit the cell below. Examples:
- **New country route** — France with leboncoin + AutoScout24
- **New source on existing route** — add `facebook_marketplace` to `us-retail`
- **Connected-only** — set `fetch_strategy="connected"`

In [ ]:
# --- EDIT THIS ---
spec = MarketSpec(
    route_slug="fr",
    route_label="France",
    country_code="FR",
    region_label="France",
    region_center=(2.3522, 48.8566),  # Paris lon, lat
    description="French classified marketplaces",
    add_to_bulk_ingest=True,
    fetch_strategy="stub",  # stub | europe | connected | custom
    sources=[
        MarketSource("leboncoin", "https://www.leboncoin.fr"),
        MarketSource("autoscout24_fr", "https://www.autoscout24.fr"),
    ],
    migration_note="Generated from notebooks/05_add_market.ipynb",
)

plan = plan_market(spec)
print_plan_summary(plan)

## 2. Preview generated SQL & snippets

In [ ]:
print("=== Migration SQL ===")
print(plan.migration_sql)
print("\n=== regions.py ===")
print(plan.regions_snippet)
print("\n=== ingest_routes.py ===")
print(plan.ingest_routes_snippet)
print(plan.bulk_routes_snippet)
print("\n=== workflow/routes.py ===")
print(plan.workflow_fetch_snippet)

## 3. Write artifacts for review

Outputs land in `notebooks/out/market_<slug>/` — copy into the repo from there.

In [ ]:
out_dir = write_artifacts(plan)
print(f"Wrote artifacts to: {out_dir}")
print("Files:", [p.name for p in sorted(out_dir.iterdir())])

## 4. Register sources in Postgres (optional)

Requires `docker compose up -d db`. Skips safely if DB is unreachable.

In [ ]:
APPLY_DB = False  # set True to upsert listings.source

if APPLY_DB:
    n = register_sources(
        settings.database_url,
        spec.sources,
        country_code=spec.country_code,
    )
    print(f"Registered {n} source(s) in listings.source")
else:
    print("Skipped DB registration (APPLY_DB=False)")

## 5. Audit — what's still missing?

In [ ]:
for check in audit_market(spec):
    mark = "✓" if check.ok else "✗"
    extra = f" ({check.detail})" if check.detail else ""
    print(f"{mark} {check.name}{extra}")

print("\nManual checklist:")
for i, item in enumerate(plan.checklist, 1):
    print(f"  {i}. {item}")

## 6. Apply migration to running DB (optional)

Copies the generated SQL from the output folder and applies via psycopg.

In [ ]:
APPLY_MIGRATION = False  # set True after reviewing SQL

if APPLY_MIGRATION:
    import psycopg
    sql_path = out_dir / plan.migration_filename
    with psycopg.connect(settings.database_url) as conn:
        conn.execute(sql_path.read_text(encoding="utf-8"))
        conn.commit()
    print(f"Applied {sql_path.name}")
    print("Remember to add the file to storage/migrations.py _MIGRATION_FILES")
else:
    print("Skipped migration apply (APPLY_MIGRATION=False)")

## Next steps

| Step | File(s) |
|------|--------|
| Migration | `db/init/<NNN>_<slug>_market.sql` + `storage/migrations.py` |
| Region map | `models/regions.py` |
| Route registry | `models/ingest_routes.py` |
| Fetch pipeline | `workflow/routes.py` + `ingestion/<slug>/` |
| Connected accounts | `ingestion/connectors.py`, `web/blueprints/public.py` |
| CLI | `scripts/ingest_<slug>.py` |
| Admin | `admin/console.py` |
| Tests | `tests/test_ingest_routes.py` |

Smoke test: `docker compose run --rm app ingest_<slug>`